In [ ]:
# Business question: Not just who churns — but when is the critical window, which segments churn fastest, 
# and what's the financial impact of each departure?

In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv('D:/Projects/Data Analytics/saas-churn-survival-analysis/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [11]:
# Check basics
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(7043, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod

In [12]:
# Fix TotalCharges — convert to numeric (blanks become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many NaN appeared
print(df['TotalCharges'].isnull().sum())
print(df[df['TotalCharges'].isnull()][['customerID','tenure','TotalCharges']])

11
      customerID  tenure  TotalCharges
488   4472-LVYGI       0           NaN
753   3115-CZMZD       0           NaN
936   5709-LVOEQ       0           NaN
1082  4367-NUYAO       0           NaN
1340  1371-DWPAZ       0           NaN
3331  7644-OMVMY       0           NaN
3826  3213-VVOLG       0           NaN
4380  2520-SGTTA       0           NaN
5218  2923-ARZLG       0           NaN
6670  4075-WKNIU       0           NaN
6754  2775-SEFEE       0           NaN


In [13]:
# Fill NaN with 0 (new customers, no charges yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Convert SeniorCitizen from 0/1 to Yes/No
df['SeniorCitizen'] = df['SeniorCitizen'].map({1:'Yes', 0:'No'})

# Convert Churn to binary
df['event'] = (df['Churn'] == 'Yes').astype(int)

# Rename tenure for clarity
df['survival_time'] = df['tenure'].clip(lower=1)

# Verify
print(df['TotalCharges'].isnull().sum())  # should be 0
print(df['event'].value_counts())
print(df[['tenure','survival_time','event']].head())

0
event
0    5174
1    1869
Name: count, dtype: int64
   tenure  survival_time  event
0       1              1      0
1      34             34      0
2       2              2      1
3      45             45      0
4       2              2      1


In [14]:
np.random.seed(42)

# Signup channel
channels = ['organic', 'paid_search', 'referral', 'social']
df['signup_channel'] = np.random.choice(channels, size=len(df),
                        p=[0.35, 0.30, 0.25, 0.10])

# Monthly revenue (slightly varied from MonthlyCharges)
df['monthly_revenue'] = (df['MonthlyCharges'] + 
                         np.random.normal(0, 2, len(df))).clip(lower=10).round(2)

# Support tickets (churned customers have more)
df['support_tickets'] = np.random.poisson(
    lam=df['event'] * 3 + (1 - df['event']) * 0.8
)

# Save cleaned file
df.to_csv('../data/processed/churn_clean.csv', index=False)
print("Saved. Shape:", df.shape)
print(df[['signup_channel','monthly_revenue','support_tickets']].head())

Saved. Shape: (7043, 26)
  signup_channel  monthly_revenue  support_tickets
0    paid_search            32.75                0
1         social            55.47                2
2       referral            55.96                4
3    paid_search            42.23                1
4        organic            72.24                4


In [15]:
print("=== Cleaning Summary ===")
print(f"Total customers: {len(df)}")
print(f"Churn rate: {df['event'].mean()*100:.1f}%")
print(f"Columns: {df.shape[1]}")
print(f"Nulls remaining: {df.isnull().sum().sum()}")

=== Cleaning Summary ===
Total customers: 7043
Churn rate: 26.5%
Columns: 26
Nulls remaining: 0
